# 02 — Llama plan executor (Colab T4)

Run **`musicinstruct run-plan-executor`** on the test split with **CUDA**. Restores dataset + prior results from Drive; writes predictions incrementally and syncs back to Drive.

**Self-contained:** clones repo, installs `.[llm]`, mounts Drive. Keep **`DATASET_MODE`** and **`RUN_ID`** aligned with notebook **01**.

**Timing (T4):** pilot ~30–60 min (50 test items); real v0.2 ~2–3 h (~180 subsampled test items).

In [ ]:
# @title Configuration
MUSICINSTRUCT_REPO = "https://github.com/juliagsy/musicinstruct.git"
BRANCH = "main"  # @param {type:"string"}

WORK = "/content"
MUSICINSTRUCT = f"{WORK}/musicinstruct"

DATASET_MODE = "real_v0.2"  # @param ["pilot", "real_v0.2"]
RUN_ID = "real_v0.2_v1"  # @param {type:"string"}  use pilot_v1 for pilot (already scored — do not rerun)

LOCAL_DATA = f"{MUSICINSTRUCT}/data/pilot" if DATASET_MODE == "pilot" else f"{MUSICINSTRUCT}/data/v0.2"
MANIFEST_NAME = "pilot.jsonl" if DATASET_MODE == "pilot" else "manifest.jsonl"
MANIFEST = f"{LOCAL_DATA}/{MANIFEST_NAME}"

MODEL = "meta-llama/Llama-3.2-1B-Instruct"  # @param {type:"string"}
SPLIT = "test"  # @param ["test", "validation"]
MAX_ITEMS = None  # @param {type:"raw"}  None = all pending items this run; e.g. 5 for smoke
MAX_NEW_TOKENS = 256  # @param {type:"integer"}  128 truncates long track lists
RESUME = True  # @param {type:"boolean"}  skip items already in predictions.jsonl

USE_DRIVE = True  # @param {type:"boolean"}
DRIVE_ROOT = "/content/drive/MyDrive/musicinstruct"
DRIVE_DATA = f"{DRIVE_ROOT}/data/pilot" if DATASET_MODE == "pilot" else f"{DRIVE_ROOT}/data/v0.2"
DRIVE_RESULTS = f"{DRIVE_ROOT}/runs/{RUN_ID}/plan_executor_llama"
LOCAL_RESULTS = f"{MUSICINSTRUCT}/results/plan_executor_llama"

In [ ]:
# @title Bootstrap (Drive, clone, install)
import os
import shutil
import subprocess
from pathlib import Path


def sh(cmd: str) -> None:
    subprocess.run(cmd, shell=True, check=True)


if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    print("Drive results:", DRIVE_RESULTS)
else:
    print("Drive disabled — results stay on this VM.")

if Path(MUSICINSTRUCT).is_dir():
    shutil.rmtree(MUSICINSTRUCT)
sh(f"git clone --branch {BRANCH} --depth 1 {MUSICINSTRUCT_REPO} {MUSICINSTRUCT}")

os.chdir(MUSICINSTRUCT)
sh('pip install -q -U "transformers>=4.44" accelerate huggingface_hub')
sh('pip install -q -e ".[llm]"')

if not os.environ.get("HF_TOKEN"):
    from getpass import getpass

    os.environ["HF_TOKEN"] = getpass("HF token (hf_...): ")

from huggingface_hub import login

login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)

In [ ]:
# @title Restore dataset & prior results from Drive
import shutil
from pathlib import Path

local_data = Path(LOCAL_DATA)
local_results = Path(LOCAL_RESULTS)
local_results.mkdir(parents=True, exist_ok=True)

if USE_DRIVE:
    drive_manifest = Path(DRIVE_DATA) / MANIFEST_NAME
    if not drive_manifest.is_file():
        raise FileNotFoundError(
            f"No {MANIFEST_NAME} on Drive at {DRIVE_DATA}. "
            f"Run notebook 01 with DATASET_MODE={DATASET_MODE!r} first."
        )
    local_data.mkdir(parents=True, exist_ok=True)
    shutil.copytree(DRIVE_DATA, local_data, dirs_exist_ok=True)
    print(f"Restored {DATASET_MODE} data from Drive:", DRIVE_DATA)

    drive_results = Path(DRIVE_RESULTS)
    if drive_results.is_dir() and any(drive_results.iterdir()):
        shutil.copytree(drive_results, local_results, dirs_exist_ok=True)
        pred = local_results / "predictions.jsonl"
        n = len(pred.read_text().splitlines()) if pred.is_file() else 0
        print(f"Restored prior results ({n} predictions)")
else:
    if not Path(MANIFEST).is_file():
        raise FileNotFoundError(
            f"Missing {MANIFEST}. Enable USE_DRIVE or run notebook 01 first."
        )

print("dataset_mode:", DATASET_MODE)
print("manifest:", MANIFEST)
print("output:", LOCAL_RESULTS)

In [ ]:
# @title GPU check
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Enable GPU: Runtime → Change runtime type → T4 GPU")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# @title Run plan executor (single session — model loaded once)
import json
import os
from pathlib import Path

os.chdir(MUSICINSTRUCT)
manifest_rel = str(Path(MANIFEST).relative_to(Path(MUSICINSTRUCT)))
cmd = [
    "musicinstruct",
    "run-plan-executor",
    manifest_rel,
    "--split",
    SPLIT,
    "--device",
    "cuda",
    "--model",
    MODEL,
    "--output-dir",
    str(Path(LOCAL_RESULTS).relative_to(Path(MUSICINSTRUCT))),
    "--max-new-tokens",
    str(MAX_NEW_TOKENS),
]
if MAX_ITEMS is not None:
    cmd.extend(["--max-items", str(MAX_ITEMS)])
if RESUME:
    cmd.append("--resume")

print(" ".join(cmd))
proc = subprocess.run(cmd, capture_output=True, text=True, check=False)
print(proc.stdout)
if proc.stderr:
    print(proc.stderr)

summary = json.loads(proc.stdout)
if "error" in summary:
    raise RuntimeError(summary["error"])

failures = summary.get("batch_failure_count", 0)
print("coverage:", summary.get("prediction_coverage"))
print("overall:", summary.get("overall"))
if failures:
    print(f"warning: {failures} item(s) failed plan/execute — see batch_failures in report")
elif proc.returncode != 0:
    raise RuntimeError(f"run-plan-executor failed (exit {proc.returncode})")

In [ ]:
# @title Progress summary
import json
from pathlib import Path

out = Path(LOCAL_RESULTS)
pred_path = out / "predictions.jsonl"
if pred_path.is_file():
    preds = [json.loads(line) for line in pred_path.read_text().splitlines() if line.strip()]
    ok = sum(1 for p in preds if p.get("midi_path"))
    fail = len(preds) - ok
    print(f"predictions: {len(preds)} ({ok} with MIDI, {fail} failed)")
    for p in preds:
        err = p.get("metadata", {}).get("error")
        status = "ok" if p.get("midi_path") else f"FAIL: {err}"
        print(f"  {p['item_id']}: {status}")
else:
    print("No predictions.jsonl yet")

report = out / "plan_executor_report.json"
if report.is_file():
    data = json.loads(report.read_text())
    print("\noverall:", data.get("overall"))
    if data.get("composition_macro"):
        print("composition_macro:", data.get("composition_macro"))

In [ ]:
# @title Sync results to Google Drive
import shutil
from pathlib import Path

local_results = Path(LOCAL_RESULTS)
if USE_DRIVE:
    Path(DRIVE_RESULTS).parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(local_results, DRIVE_RESULTS, dirs_exist_ok=True)
    print("Synced →", DRIVE_RESULTS)
else:
    print("Skipped Drive sync. Local:", local_results)

In [ ]:
# @title Download results zip (optional)
import shutil
from pathlib import Path

zip_base = f"{WORK}/plan_executor_{RUN_ID}"
shutil.make_archive(zip_base, "zip", LOCAL_RESULTS)
try:
    from google.colab import files

    files.download(zip_base + ".zip")
except ImportError:
    print("Zip at:", zip_base + ".zip")